# Brain Space Visualization of Deviation Scores

In this tutorial, our objective is to identify regions in the brain with significant positive and negative deviations. To accomplish this, we load the previously calculated z-scores for each participant and each ROI. We can then create individual plots to visualize the deviations for each participant in different ROIs, or summary plots by aggregating the deviations for a group of individuals.

Specifically, in this tutorial, we will:

1. Determine the number of extreme deviations (|Z|>2) in each brain region.
2. Visualize the count of extreme deviations for each hemisphere.

By following these steps, we can gain insights into the distribution of deviations and identify brain regions that exhibit remarkable positive or negative deviations.

To learn more about extreme deviations in normative modeling refer to this [paper](https://www.biorxiv.org/content/10.1101/2022.08.23.505049v3)

In [ ]:
! git clone https://github.com/CharFraza/CPC_ML_tutorial.git

In [ ]:
%%capture
! pip install nilearn

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from nilearn import plotting
import nibabel as nib
from nilearn import datasets
import statsmodels.api as sm

In [ ]:
sns.set(style='whitegrid')
sns.despine()
sns.set_palette("pastel") # play around with different color palettes: "deep", "muted", "bright", "pastel", "dark", and "colorblind"


In [ ]:
os.chdir('/content/CPC_ML_tutorial')

In [ ]:
Z_df = pd.read_csv('data/Z_long_format.csv')

In [ ]:
Z_df.head()

In [ ]:
plt.figure()
sns.histplot(Z_df.value, bins=20)
plt.title('Z-scores')

In [ ]:
# Create a QQ plot of the Z_df scores
sm.qqplot(Z_df['value'], line='s')
plt.title('QQ Plot of Z-scores')
plt.show()

1. Change this threshold to view more or less extreme deviations.
2. Discuss with your partner what you think is an appropriate threshold and adjust the variables below accordingly.

In [ ]:
Z_positive = Z_df.query('value > 2')
Z_negative = Z_df.query('value < -2')

In [ ]:
positive_left_z = Z_positive.query('hemi == "left"')
positive_right_z = Z_positive.query('hemi == "right"')
positive_sc_z = Z_positive.query('hemi == "subcortical"')
negative_left_z = Z_negative.query('hemi == "left"')
negative_right_z = Z_negative.query('hemi == "right"')
negative_sc_z = Z_negative.query('hemi == "subcortical"')

In [ ]:
positive_left_z2 = positive_left_z['ROI_name'].value_counts().rename_axis('ROI').reset_index(name='counts')
positive_right_z2 = positive_right_z['ROI_name'].value_counts().rename_axis('ROI').reset_index(name='counts')
positive_sc_z2 = positive_sc_z['ROI_name'].value_counts().rename_axis('ROI').reset_index(name='counts')
negative_left_z2 = negative_left_z['ROI_name'].value_counts().rename_axis('ROI').reset_index(name='counts')
negative_right_z2 = negative_right_z['ROI_name'].value_counts().rename_axis('ROI').reset_index(name='counts')
negative_sc_z2 = negative_sc_z['ROI_name'].value_counts().rename_axis('ROI').reset_index(name='counts')

In [ ]:
positive_left_z2.head()

In [ ]:
plt.figure()
sns.histplot(positive_left_z2.counts, bins=20)
plt.title('Number of extreme positive deviations per participant')

In [ ]:
positive_right_z2.describe()

In [ ]:
positive_sc_z2.describe()

In [ ]:
negative_left_z2.describe()

In [ ]:
negative_right_z2.describe()

In [ ]:
negative_sc_z2.describe()

Download and load the Destrieux atlas of cortical regions from the nilearn library's datasets module.

In [ ]:
%%capture
destrieux_atlas = datasets.fetch_atlas_surf_destrieux()
fsaverage = datasets.fetch_surf_fsaverage()

In [ ]:
# The parcellation is already loaded into memory
parcellation_l = destrieux_atlas['map_left']
parcellation_r = destrieux_atlas['map_right']

In [ ]:
nl = pd.read_csv('data/nilearn_order.csv')

In [ ]:
atlas_r = destrieux_atlas['map_right']
atlas_l = destrieux_atlas['map_left']

In [ ]:
nl_ROI = nl['ROI'].to_list()

# Extreme positive deviation visualization

In [ ]:
nl_positive_left = pd.merge(nl, positive_left_z2, on='ROI', how='left')
nl_positive_right = pd.merge(nl, positive_right_z2, on='ROI', how='left')

In [ ]:
nl_positive_left['counts'] = nl_positive_right['counts'].fillna(0)
nl_positive_right['counts'] = nl_positive_right['counts'].fillna(0)

In [ ]:
nl_positive_left = nl_positive_left['counts'].to_numpy()
nl_positive_right = nl_positive_right['counts'].to_numpy()

In [ ]:
a_list = list(range(1, 76))
parcellation_positive_l = atlas_l
for i, j in enumerate(a_list):
    parcellation_positive_l = np.where(parcellation_positive_l == j, nl_positive_left[i], parcellation_positive_l)

In [ ]:
a_list = list(range(1, 76))
parcellation_positive_r = atlas_r
for i, j in enumerate(a_list):
    parcellation_positive_r = np.where(parcellation_positive_r == j, nl_positive_right[i], parcellation_positive_r)

In [ ]:
# you can click around in 3D space on this visualization. Scroll in/out, move the brain around, etc. Have fun with it :)
view = plotting.view_surf(fsaverage.infl_right, parcellation_positive_r, threshold=None, symmetric_cmap=False, cmap='plasma', bg_map=fsaverage.sulc_right)

view

In [ ]:
view = plotting.view_surf(fsaverage.infl_left, parcellation_positive_l, threshold=None, symmetric_cmap=False, cmap='plasma', bg_map=fsaverage.sulc_left)

view

# Extreme negative deviation visualization

In [ ]:
nl_negative_left = pd.merge(nl, negative_left_z2, on='ROI', how='left')
nl_negative_right = pd.merge(nl, negative_right_z2, on='ROI', how='left')

In [ ]:
nl_negative_left['counts'] = nl_negative_left['counts'].fillna(0)
nl_negative_right['counts'] = nl_negative_right['counts'].fillna(0)

In [ ]:
nl_negative_left = nl_negative_left['counts'].to_numpy()
nl_negative_right = nl_negative_right['counts'].to_numpy()

In [ ]:
a_list = list(range(1, 76))
parcellation_negative_l = atlas_l
for i, j in enumerate(a_list):
    parcellation_negative_l = np.where(parcellation_negative_l == j, nl_negative_left[i], parcellation_negative_l)

In [ ]:
a_list = list(range(1, 76))
parcellation_negative_r = atlas_r
for i, j in enumerate(a_list):
    parcellation_negative_r = np.where(parcellation_negative_r == j, nl_negative_right[i], parcellation_negative_r)

In [ ]:
view = plotting.view_surf(fsaverage.infl_right, parcellation_negative_r, threshold=None, symmetric_cmap=False, cmap='plasma', bg_map=fsaverage.sulc_right)

view

In [ ]:
view = plotting.view_surf(fsaverage.infl_left, parcellation_negative_l, threshold=None, symmetric_cmap=False, cmap='plasma', bg_map=fsaverage.sulc_left)

view

## Questions
1. How does the inclusion of different covariates impact the calculation of z-scores in normative models?
2. Are there significant differences in z-scores between 'healthy' participants and participants with conditions such as Alzheimer's? If yes, how and in which brain regions do these differences manifest?
3. Given that normative models generate multiple z-scores per individual, what approach can a clinician adopt to derive a single health score for an individual based on these z-scores?

## Suggested further readings
1. [Fairness: Types of Bias](https://developers.google.com/machine-learning/crash-course/fairness/types-of-bias)
2. [Controlling for effects of confounding variables on machine learning predictions](https://www.biorxiv.org/content/10.1101/2020.08.17.255034v1.abstract)
